> Disclaimer
>
> This notebook is not intended to be used and was created in the process of developping Bob.
> Some or many features may have been modified since the creation of this file.
> Use at your own risk. 
>
> Christian Tremblay

In [3]:
from bob.core import (
    bind_model_namespace,
    Equipment,
    enum,
    Junction,
    Air,
    dump,
    p223
)
from rdflib import URIRef

from bob.space.hvac import HVACSpace, HVACZone
from bob.space.physical import Building, Floor, MechanicalRoom, Office
from bob.equipment.hvac.fan import Fan
from bob.connections.light import Light, LightConnection, LightVisible, LightVisibleOutletConnectionPoint
from bob.connections.electricity import ElectricalInletConnectionPoint
from pathlib import Path


model_name = Path('Notebook').stem
_namespace = bind_model_namespace("ex", f"urn:ex/{model_name}/")



In [6]:

class WeirdLuminaire(Equipment):
    node_type: URIRef = p223.Light
    lightOutlet: LightVisibleOutletConnectionPoint
    electricalInlet: ElectricalInletConnectionPoint

lum = WeirdLuminaire(label='Sun')
lightcnx = LightConnection(label='Rx')

lum >> lightcnx

dump()


RuntimeError: no common connection types

In [10]:
from bob.connections.water import WaterConnection, ChilledWaterOutletConnectionPoint

In [20]:
from typing import Any
from collections import defaultdict
from bob.core import *

def connect(from_thing: Any, to_thing: Any, segmented: bool = False) -> None:
    """
    Find an unambiguous way to connect to things together.
    """
    logging.info(f"connect from {from_thing} to {to_thing}")

    from_out = defaultdict(set)
    if isinstance(from_thing, (Connection, ConnectionPoint)):
        medium = getattr(from_thing, "hasMedium", None)
        # medium = getattr(medium, "node", medium)
        from_out[medium].add(from_thing)

    elif isinstance(from_thing, Connectable):
        for attr, connection_point in from_thing._connection_points.items():
            if connection_point.connectsThrough:
                continue
            if not isinstance(connection_point, OutletConnectionPoint):
                continue

            medium = getattr(connection_point, "hasMedium", None)
            # medium = getattr(medium, "node", medium)
            # ISSUE...having a hard time with electrical things
            from_out[medium].add(connection_point)

    elif isinstance(from_thing, (SystemConnectionPoint, ZoneConnectionPoint)):
        if not from_thing.mapsTo:
            if isinstance(from_thing, SystemConnectionPoint):
                raise RuntimeError(f"unmapped system connection point {to_thing}")
            if isinstance(from_thing, ZoneConnectionPoint):
                raise RuntimeError(f"unmapped zone connection point {to_thing}")
        connection_point = from_thing.mapsTo

        if isinstance(connection_point, ConnectionPoint):
            if connection_point.connectsThrough:
                raise RuntimeError(
                    f"connection point already connected: {connection_point}"
                )
            if getattr(connection_point, "hasDirection", None) == Inlet:
                raise TypeError(f"connection point direction: {connection_point}")
        elif isinstance(connection_point, Junction):
            pass

        medium = getattr(connection_point, "hasMedium", None)
        from_out[medium].add(connection_point)

    elif isinstance(from_thing, System):
        for attr, connection_point in from_thing._system_connection_points.items():
            if not connection_point.mapsTo:
                continue
            connection_point = connection_point.mapsTo

            if isinstance(connection_point, ConnectionPoint):
                if connection_point.connectsThrough:
                    continue
                if getattr(connection_point, "hasDirection", None) == Inlet:
                    continue
            elif isinstance(connection_point, Junction):
                pass

            medium = getattr(connection_point, "hasMedium", None)
            from_out[medium].add(connection_point)

    elif isinstance(from_thing, Zone):
        for attr, connection_point in from_thing._zone_connection_points.items():
            if not connection_point.mapsTo:
                continue
            connection_point = connection_point.mapsTo

            if isinstance(connection_point, ConnectionPoint):
                if connection_point.connectsThrough:
                    continue
                if getattr(connection_point, "hasDirection", None) == Inlet:
                    continue
            elif isinstance(connection_point, Junction):
                pass

            medium = getattr(connection_point, "hasMedium", None)
            from_out[medium].add(connection_point)

    else:
        raise NotImplementedError(f"connecting from {from_thing}")
    logging.debug(f"    - from_out: {from_out}")

    from_types: Set[Medium]
    if isinstance(from_thing, Connection):
        from_types = set([from_thing.hasMedium])
    else:
        from_types = set(medium for medium in from_out if len(from_out[medium]) == 1)
        if not from_types:
            raise RuntimeError(
                f"no candidate sources from {from_thing.node} to {to_thing.node}"
            )
    logging.debug(f"    - from_types: {from_types}")

    to_in = defaultdict(set)
    if isinstance(to_thing, (Connection, ConnectionPoint)):
        medium = getattr(to_thing, "hasMedium", None)
        # medium = getattr(medium, "node", medium)
        # ISSUE...having a hard time with electrical things
        # maybe this was due to me, breaking Joel's toy
        to_in[medium].add(to_thing)

    elif isinstance(to_thing, Connectable):
        for attr, connection_point in to_thing._connection_points.items():
            if connection_point.connectsThrough:
                continue
            if not isinstance(connection_point, InletConnectionPoint):
                continue

            medium = getattr(connection_point, "hasMedium", None)
            # Here when trying to connect a connectionpoint to a Equipment
            # medium turned to be
            # {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Water-ChilledWater'), 'label': '', 'comment': ''}
            # and the intersection fails to recognize the substance
            # medium = getattr(medium, "node", medium)
            to_in[medium].add(connection_point)

    elif isinstance(to_thing, (SystemConnectionPoint, ZoneConnectionPoint)):
        if not to_thing.mapsTo:
            if isinstance(to_thing, SystemConnectionPoint):
                raise RuntimeError(f"unmapped system connection point {to_thing}")
            if isinstance(to_thing, ZoneConnectionPoint):
                raise RuntimeError(f"unmapped zone connection point {to_thing}")
        connection_point = to_thing.mapsTo

        if isinstance(connection_point, ConnectionPoint):
            if connection_point.connectsThrough:
                raise RuntimeError(
                    f"connection point already connected: {connection_point}"
                )
            if getattr(connection_point, "hasDirection", None) == Outlet:
                raise TypeError(f"connection point direction: {connection_point}")
        elif isinstance(connection_point, Junction):
            pass

        medium = getattr(connection_point, "hasMedium", None)
        to_in[medium].add(connection_point)

    elif isinstance(to_thing, System):
        for attr, connection_point in to_thing._system_connection_points.items():
            if not connection_point.mapsTo:
                continue
            connection_point = connection_point.mapsTo

            if isinstance(connection_point, ConnectionPoint):
                if connection_point.connectsThrough:
                    continue
                if getattr(connection_point, "hasDirection", None) == Outlet:
                    continue
            elif isinstance(connection_point, Junction):
                pass

            medium = getattr(connection_point, "hasMedium", None)
            to_in[medium].add(connection_point)

    elif isinstance(to_thing, Zone):
        for attr, connection_point in to_thing._zone_connection_points.items():
            if not connection_point.mapsTo:
                continue
            connection_point = connection_point.mapsTo

            if isinstance(connection_point, ConnectionPoint):
                if connection_point.connectsThrough:
                    continue
                if getattr(connection_point, "hasDirection", None) == Outlet:
                    continue
            elif isinstance(connection_point, Junction):
                pass

            medium = getattr(connection_point, "hasMedium", None)
            to_in[medium].add(connection_point)

    else:
        raise NotImplementedError(f"connecting to {to_thing}")
    logging.debug(f"    - to_in: {to_in}")

    to_types: Set[Medium]
    if isinstance(to_thing, Connection):
        to_types = set([to_thing.hasMedium])
    else:
        to_types = set(medium for medium in to_in if len(to_in[medium]) == 1)
        if not to_types:
            raise RuntimeError(
                f"no candidate destinations from {from_thing.node} to {to_thing.node}"
            )
    logging.debug(f"    - to_types: {to_types}")

    # find the common medium
    common_types = from_types.intersection(to_types)
    if not common_types:
        #raise RuntimeError("no common connection types")
        return (_, from_types, _, to_types)
    if len(common_types) > 1:
        raise RuntimeError("too many common connection types")
    medium = common_types.pop()
    logging.debug(f"    - medium: {medium}")

    if isinstance(from_thing, Connection):
        if isinstance(to_thing, Connection):
            raise RuntimeError("connection to connection")
        to_connection_point = to_in[medium].pop()

        from_thing.connect_to(to_connection_point)

    elif isinstance(to_thing, Connection):
        from_connection_point = from_out[medium].pop()

        to_thing.connect_from(from_connection_point)

    else:
        # get the medium and the two connection points
        from_connection_point = from_out[medium].pop()
        to_connection_point = to_in[medium].pop()

        # if either connection point is a junction, this is segmented
        if (
            segmented
            or isinstance(from_connection_point, Junction)
            or isinstance(to_connection_point, Junction)
        ):
            segment = Segment()
            segment.link_to(from_connection_point)
            segment.link_to(to_connection_point)
        else:
            from_connection_point.connect_to(to_connection_point)

    return (from_out, from_types, to_in, to_types)


In [40]:
wc = WaterConnection(label='wc')
class Chiller(Equipment):
    out: ChilledWaterOutletConnectionPoint

chiller = Chiller(label='Chiller')
fo, ft, to, tt = connect(chiller, wc)

In [41]:
_

{<Medium at http://data.ashrae.org/standard223#Water-ChilledWater>}

In [42]:
m.__dir__

<function Medium.__dir__()>

In [23]:
tp

{<Medium at http://data.ashrae.org/standard223#Medium-Water>}

In [24]:
from bob.connections.water import WaterInletConnectionPoint, WaterOutletConnectionPoint
class Valve(Equipment):
    waterInlet: WaterInletConnectionPoint
    waterOutlet: WaterOutletConnectionPoint

In [29]:
v = Valve(label='SuperValve')

In [26]:
v

<Valve SuperValve at urn:ex/Notebook/00027>

In [30]:
dump()

@prefix ex: <urn:ex/Notebook/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
ex:00001 a s223:Equipment,
        ex:Valve ;
    rdfs:label "SuperValve" ;
    s223:hasConnectionPoint ex:00002,
        ex:00003 ;
    ex:waterInlet ex:00002 ;
    ex:waterOutlet ex:00003 .
ex:00002 a s223:ConnectionPoint,
        s223:InletConnectionPoint ;
    rdfs:label "SuperValve.waterInlet" ;
    s223:hasDirection s223:Direction-Inlet ;
    s223:hasMedium s223:Medium-Water .
ex:00003 a s223:ConnectionPoint,
        s223:OutletConnectionPoint ;
    rdfs:label "SuperValve.waterOutlet" ;
    s223:hasDirection s223:Direction-Outlet ;
    s223:hasMedium s223:Medium-Water .
